##**1. Data Munging** -

**1. Visibily/Manually opening the file and capture couple of data patterns (Manual Exploratory Data Analysis)**

In [0]:
df = spark.read.option("multiline", "true").json("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_shipment_detail_3000.json")
display(df)
df.printSchema()

####2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
1. Apply inferSchema and toDF to create a DF and analyse the actual data.
2. Analyse the schema, datatypes, columns etc.,
3. Analyse the duplicate records count and summary of the dataframe.

In [0]:
df_ls1=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source1")
display(df_ls1)
df_ls1.printSchema()
display(df.summary())
display(df.describe())

###a. Passive Data Munging -  (File: logistics_source1  and logistics_source2)
Without modifying the data, identify:<br>
Shipment IDs that appear in both master_v1 and master_v2<br>
Records where:<br>
1. shipment_id is non-numeric
2. age is not an integer<br>

Count rows having:
3. fewer columns than expected
4. more columns than expected

In [0]:
from pyspark.sql.functions import col
df_ls1=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source1")
df_ls2=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source2")
# df_ls1.printSchema()
# df_ls2.printSchema()
# display(df_ls1)
# display(df_ls2)
# display(df_ls1.join(df_ls2.withColumn('shipment_id',col('shipment_id').cast('string')), how='inner', on='shipment_id'))#1

# df_leftout=df_ls1.join(df_ls2.withColumn('shipment_id',col('shipment_id').cast('string')), how='left', on='shipment_id')
display(df_ls1.where("age not rlike '[^0-9]'"))


###**b. Active Data Munging** File: logistics_source1 and logistics_source2

%md
#####1.Combining Data + Schema Merging (Structuring)
1. Read both files without enforcing schema
2. Align them into a single canonical schema: shipment_id,
first_name,
last_name,
age,
role,
hub_location,
vehicle_type,
data_source
3. Add data_source column with values as: system1, system2 in the respective dataframes

In [0]:
# List of source files
file_paths = [
    "/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source1", 
    "/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source2"
]

# Read without a strict schema
df = (spark.read
      .option("header", "true")       # Uses the first row as column names
      .option("inferSchema", "true")  # Automatically detects data types
      .csv(file_paths))

# Display the results for manual inspection
display(df)

In [0]:
from pyspark.sql.functions import col,lit,when,expr

df_ls1=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source1")
df_ls2=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source2")

df_ls1_T1=df_ls1.withColumn("shipment_id",when(col("shipment_id")=="null",lit(0)).
                               when(col("shipment_id")=="ten",lit(10) ).otherwise(col("shipment_id") ))

df_ls1_T2=(df_ls1_T1.withColumn("Source",lit('Master1'))).withColumn("shipment_id",col("shipment_id").cast('integer'))

# df_ls1_T2.where("age not rlike '[0-9]'").show(truncate=False)

df_mas1=df_ls1_T2.withColumn("age",when(col("age")=="null",lit(0)).
                                when(col("age")=="ten",lit(10) ).otherwise(col("age") ))


# display(df_mas1)
# df_mas1.printSchema()
df_mas2_temp =df_ls2.withColumn("Source",lit('Master2'))
df_mas2=df_mas2_temp.withColumn("age",when(col("age")=="null",lit(0)).
                                when(col("age")=="ten",lit(10) ).otherwise(col("age") ))

# display(df_mas2)
canonical_df = df_mas1.unionByName(df_mas2, allowMissingColumns=True)
# canonical_df = canonical_df.select(
#     "shipment_id", "first_name", "last_name", "age", 
#     "role", "hub_location", "vehicle_type", "Source"
# )
 
display(canonical_df)
 

#####2. Cleansing, Scrubbing: 
Cleansing (removal of unwanted datasets)<br>
1. Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role<br>
2. Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name<br>
3. Join Readiness Rule - Drop records where the join key is null: shipment_id<br>

Scrubbing (convert raw to tidy)<br>
4. Age Defaulting Rule - Fill NULL values in the age column with: -1<br>
5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN<br>
6. Invalid Age Replacement - Replace the following values in age:
"ten" to -1
"" to -1<br>
7. Vehicle Type Normalization - Replace inconsistent vehicle types: 
truck to LMV
bike to TwoWheeler

In [0]:
# display(canonical_df)
# drop_sr_df=canonical_df.dropna(subset=["shipment_id", "role"])
# display(drop_sr_df)
# drop_fl_df=canonical_df.dropna(subset=["first_name","last_name"])
# display(drop_fl_df)
# drop_sid_df=canonical_df.dropna(subset=["shipment_id"])
# display(drop_sid_df)
# agenull_df=canonical_df.withColumn("age",when(col("age").isNull(),lit("-1")).
#                                 otherwise(col("age")   ))
agenull_df=canonical_df.fillna(value=-1, subset=["age"])
display(agenull_df)                                

####3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

Detail Dataframe creation <br>
1. Read Data from logistics_shipment_detail.json
2. As this data is a clean json data, it doesn't require any cleansing or scrubbing.


Standardizations:<br>

1. Add a column<br> 
Source File: logistics_shipment_detail_3000.json<br>: domain as 'Logistics'
2. Column Uniformity: 
role - Convert to lowercase<br>
Source File: logistics_source1 & logistics_source2<br>
vehicle_type - Convert values to UPPERCASE<br>
Source Files: logistics_shipment_detail_3000.json (and the merged master files)
hub_location - Convert values to initcap case<br>
3. Format Standardization:<br>
Source Files: logistics_shipment_detail_3000.json
Convert shipment_ref to string<br>
Pad to 10 characters with leading zeros<br>
Convert dispatch_date to yyyy-MM-dd<br>
Ensure delivery_cost has 2 decimal precision<br>
4. Data Type Standardization<br>
Standardizing column data types to fix schema drift and enable mathematical operations.<br>
Source File: logistics_source1 & logistics_source2 <br>
age: Cast String to Integer<br>
Source File: logistics_shipment_detail_3000.json<br>
shipment_weight_kg: Cast to Double<br>
Source File: logistics_shipment_detail_3000.json<br>
is_expedited: Cast to Boolean<br>
5. Naming Standardization <br>
Source File: logistics_source1 & logistics_source2<br>
Rename: first_name to staff_first_name<br>
Rename: last_name to staff_last_name<br>
Rename: hub_location to origin_hub_city<br>
6. Reordering columns logically in a better standard format:<br>
Source File: All 3 files<br>
shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)


In [0]:
from pyspark.sql.functions import lit, col, upper,to_date

df_json = spark.read.option("multiline", "true").json("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_shipment_detail_3000.json")
# df_json_Add = df_json.withColumn("domain ", lit("Logistics")).withColumn("vehicle_type", upper(col("vehicle_type")))
df_json_Add = df_json.withColumn("shipment_date",to_date(col("shipment_date"),"dd-MM-yy"))
display(df_json_Add.take(10))
display(df_json.take(10))

##2. Data Enrichment - Detailing of data
Makes your data rich and detailed <br>


###### Adding of Columns (Data Enrichment)
*Creating new derived attributes to enhance traceability and analytical capability.*

**1. Add Audit Timestamp (`load_dt`)**
Source File: logistics_source1 and logistics_source2<br>
* **Scenario:** We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
* **Action:** Add a column `load_dt` using the function `current_timestamp()`.

**2. Create Full Name (`full_name`)**
Source File: logistics_source1 and logistics_source2<br>
* **Scenario:** The reporting dashboard requires a single field for the driver's name instead of separate columns.
* **Action:** Create `full_name` by concatenating `first_name` and `last_name` with a space separator.
* **Result:** "Rajesh" + " " + "Kumar" -> **"Rajesh Kumar"**

**3. Define Route Segment (`route_segment`)**
Source File: logistics_shipment_detail_3000.json<br>
* **Scenario:** The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
* **Action:** Combine `source_city` and `destination_city` with a hyphen.
* **Result:** "Chennai" + "-" + "Pune" -> **"Chennai-Pune"**

**4. Generate Vehicle Identifier (`vehicle_identifier`)**
Source File: logistics_shipment_detail_3000.json<br>
* **Scenario:** We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
* **Action:** Combine `vehicle_type` and `shipment_id` to create a composite key.
* **Result:** "Truck" + "_" + "500001" -> **"Truck_500001"**


In [0]:
from pyspark.sql.functions import col,lit,when,expr,concat
from pyspark.sql import functions as F

# df_ls1=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source1")
# df_ls2=spark.read.options(header='true', inferSchema='true').csv("/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source2")
# display(df_ls1.withColumn("load_dt",F.current_timestamp()))

# List of source files
file_paths = [
    "/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source1", 
    "/Volumes/logisticscatalog47/logisticsschemaw47/logsisticvol47/logistics_source2"
]

# Read without a strict schema
df = (spark.read
      .option("header", "true")       # Uses the first row as column names
      .option("inferSchema", "true")  # Automatically detects data types
      .csv(file_paths))

# Display the results for manual inspection
df_fulNm=df.withColumn("fullname",concat(
     col("first_name"),lit(" "),col("last_name")
    ))

# df = df.withColumn("load_dt",F.current_timestamp()) "),col("last_name"))))    
display(df_fulNm)
